# NYC Taxi Fare Prediction using Artificial Neural Network

This notebook demonstrates a complete machine learning workflow for predicting taxi fare amounts using the NYC Taxi Fare Prediction dataset with an Artificial Neural Network (ANN) model built using scikit-learn.

## 1. Objective

The goal of this assignment is to:
- **Preprocess and clean** the NYC Taxi Fare dataset, handling missing values and outliers
- **Engineer features** from pickup datetime and geographic coordinates
- **Train an Artificial Neural Network** (MLP Regressor) for fare amount prediction
- **Evaluate** the model using appropriate metrics (MAE, RMSE, R²)
- **Deploy** the model as an interactive Gradio application on Hugging Face Spaces

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Print versions for reproducibility
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

import sklearn

## 2. Data Loading and Exploration

The NYC Taxi Fare Prediction dataset contains historically accurate taxi trip data with fare amounts. Let's load and explore the data.

In [ ]:
# Load the dataset
df = pd.read_csv('data/nyc_taxi_fare.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

## 3. Data Preprocessing and Cleaning

We need to clean the data by:
- Removing null values
- Filtering out unrealistic fare amounts
- Validating geographic coordinates (must be within NYC bounds)
- Ensuring pickup and dropoff are different locations

In [ ]:
# Data cleaning
rows_before = len(df)

# Convert datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')

# Remove null values
required_cols = ['pickup_datetime', 'pickup_longitude', 'pickup_latitude', 
                 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'fare_amount']
df = df.dropna(subset=required_cols)

# Filter fare amounts (realistic range)
df = df[(df['fare_amount'] >= 0) & (df['fare_amount'] <= 500)]

# Filter passenger count (realistic range)
df = df[(df['passenger_count'] >= 1) & (df['passenger_count'] <= 8)]

# NYC geographic bounds
df = df[(df['pickup_latitude'] >= 40.0) & (df['pickup_latitude'] <= 42.0)]
df = df[(df['pickup_longitude'] >= -75.5) & (df['pickup_longitude'] <= -72.5)]
df = df[(df['dropoff_latitude'] >= 40.0) & (df['dropoff_latitude'] <= 42.0)]
df = df[(df['dropoff_longitude'] >= -75.5) & (df['dropoff_longitude'] <= -72.5)]

# Ensure pickup and dropoff are different
df = df[(df['pickup_latitude'] != df['dropoff_latitude']) | 
        (df['pickup_longitude'] != df['dropoff_longitude'])]

rows_after = len(df)
print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")
print(f"Data shape: {df.shape}")

## 4. Feature Engineering

We'll engineer features from the raw data:
- **Time-based features**: hour, day of week, month, year, weekend flag
- **Distance features**: Haversine distance between pickup and dropoff
- **Location features**: Absolute differences in latitude and longitude

In [ ]:
# Feature Engineering

# Time-based features
df['pickup_hour'] = df['pickup_datetime'].dt.hour.fillna(0).astype(int)
df['pickup_day_of_week'] = df['pickup_datetime'].dt.dayofweek.fillna(0).astype(int)
df['pickup_month'] = df['pickup_datetime'].dt.month.fillna(0).astype(int)
df['pickup_year'] = df['pickup_datetime'].dt.year.fillna(0).astype(int)
df['is_weekend'] = (df['pickup_datetime'].dt.dayofweek >= 5).astype(int)

# Haversine distance calculation
def haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0088
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad
    a = np.sin(delta_lat/2)**2 + np.cos(lat1_rad)*np.cos(lat2_rad)*np.sin(delta_lon/2)**2
    return 2 * earth_radius_km * np.arcsin(np.sqrt(a))

df['trip_distance_km'] = haversine_km(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

# Geographic features
df['abs_lat_diff'] = (df['pickup_latitude'] - df['dropoff_latitude']).abs()
df['abs_lon_diff'] = (df['pickup_longitude'] - df['dropoff_longitude']).abs()

print("Features created successfully!")
print(f"Feature columns: {[col for col in df.columns if col not in ['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']]}")

## 5. Model Training

We'll build and train an Artificial Neural Network using scikit-learn's MLPRegressor:
- **Architecture**: 3-layer network (128 -> 64 -> 32 neurons)
- **Activation**: ReLU activation function
- **Optimizer**: Adam solver
- **Scaling**: StandardScaler for feature normalization

In [ ]:
# Select features for model training
feature_columns = ['passenger_count', 'pickup_hour', 'pickup_day_of_week', 
                   'pickup_month', 'pickup_year', 'is_weekend', 
                   'trip_distance_km', 'abs_lat_diff', 'abs_lon_diff']

X = df[feature_columns].astype(float)
y = df['fare_amount'].astype(float)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Build the ANN model using Pipeline (includes scaling)
model = Pipeline([
    ('scaler', StandardScaler()),
    ('ann', MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        solver='adam',
        alpha=0.0001,
        batch_size=1024,
        learning_rate_init=0.001,
        max_iter=120,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12,
        random_state=42,
        verbose=False
    ))
])

# Train the model
print("Training ANN model...")
model.fit(X_train, y_train)
print("✓ Model training complete!")

## 6. Model Evaluation

Let's evaluate the trained model on the test set using standard regression metrics.

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate metrics
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)

train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

# Display results
print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"\nTraining Set:")
print(f"  MAE:  ${train_mae:.4f}")
print(f"  RMSE: ${train_rmse:.4f}")
print(f"  R²:   {train_r2:.4f}")

print(f"\nTest Set:")
print(f"  MAE:  ${test_mae:.4f}")
print(f"  RMSE: ${test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")
print("=" * 50)

## 7. Visualizations

Let's visualize the model's predictions versus actual values.

In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted (Test Set)
axes[0].scatter(y_test, y_pred_test, alpha=0.5, s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Fare ($)')
axes[0].set_ylabel('Predicted Fare ($)')
axes[0].set_title(f'Actual vs Predicted Fare (Test Set)\nR² = {test_r2:.4f}')
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.5, s=10)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Fare ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Residual: ${residuals.mean():.4f}")
print(f"Std Dev Residuals: ${residuals.std():.4f}")

## 8. Make Predictions on New Data

Let's demonstrate how to make predictions for new taxi trips.

In [ ]:
# Example prediction for a new trip
from datetime import datetime

# Define a sample trip
sample_trip = pd.DataFrame({
    'pickup_datetime': [datetime(2015, 6, 15, 14, 30, 0)],
    'pickup_longitude': [-73.985],
    'pickup_latitude': [40.748],
    'dropoff_longitude': [-73.971],
    'dropoff_latitude': [40.783],
    'passenger_count': [2]
})

# Extract features for the sample trip
sample_trip['pickup_hour'] = sample_trip['pickup_datetime'].dt.hour
sample_trip['pickup_day_of_week'] = sample_trip['pickup_datetime'].dt.dayofweek
sample_trip['pickup_month'] = sample_trip['pickup_datetime'].dt.month
sample_trip['pickup_year'] = sample_trip['pickup_datetime'].dt.year
sample_trip['is_weekend'] = (sample_trip['pickup_datetime'].dt.dayofweek >= 5).astype(int)
sample_trip['trip_distance_km'] = haversine_km(
    sample_trip['pickup_latitude'], sample_trip['pickup_longitude'],
    sample_trip['dropoff_latitude'], sample_trip['dropoff_longitude']
)
sample_trip['abs_lat_diff'] = (sample_trip['pickup_latitude'] - sample_trip['dropoff_latitude']).abs()
sample_trip['abs_lon_diff'] = (sample_trip['pickup_longitude'] - sample_trip['dropoff_longitude']).abs()

# Select only the feature columns
sample_features = sample_trip[feature_columns].astype(float)

# Make prediction
predicted_fare = model.predict(sample_features)[0]

print(f"Sample Trip Details:")
print(f"  Pickup: ({sample_trip['pickup_latitude'].values[0]:.6f}, {sample_trip['pickup_longitude'].values[0]:.6f})")
print(f"  Dropoff: ({sample_trip['dropoff_latitude'].values[0]:.6f}, {sample_trip['dropoff_longitude'].values[0]:.6f})")
print(f"  Passengers: {int(sample_trip['passenger_count'].values[0])}")
print(f"  Distance: {sample_trip['trip_distance_km'].values[0]:.2f} km")
print(f"\n✓ Predicted Fare: ${predicted_fare:.2f}")

## 9. Model Persistence

Save the trained model for deployment in a Gradio interface.

In [ ]:
# Save the model using joblib
import joblib

model_path = 'artifacts/taxi_fare_ann_model.joblib'
joblib.dump(model, model_path)
print(f"✓ Model saved to: {model_path}")

# Save metrics
import json
metrics_data = {
    'mae': float(test_mae),
    'rmse': float(test_rmse),
    'r2': float(test_r2)
}

metrics_path = 'artifacts/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_data, f, indent=2)
print(f"✓ Metrics saved to: {metrics_path}")

## 10. Deployment

The trained model is deployed as an interactive Gradio web application that allows users to:
- Input trip details (pickup/dropoff locations, datetime, passenger count)
- Get real-time fare predictions
- View sample predictions

### Running the Gradio App

```bash
python app.py
```

The app will be available at `http://localhost:7860`

### Deployment to Hugging Face Spaces

To deploy on Hugging Face Spaces:
1. Create a new Space on Hugging Face (https://huggingface.co/spaces)
2. Choose "Gradio" as the SDK
3. Connect to your GitHub repository
4. Hugging Face will automatically run `app.py`

For detailed deployment instructions, see the `README.md` file.